In [84]:
import os

In [85]:
%pwd

'E:\\mlops\\end-to-end-machine-learning-project-with-mlflow'

In [86]:
# os.chdir("../") 
os.chdir("E:/mlops/end-to-end-machine-learning-project-with-mlflow")

In [87]:
%pwd

'E:\\mlops\\end-to-end-machine-learning-project-with-mlflow'

In [88]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/AdeebInamdar/end-to-end-machine-learning-project-with-mlflow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="AdeebInamdar"
os.environ["MLFLOW_TRACKING_PASSWORD"]="c284d8c9169426b9727cd7cebdfd2a1485f6cb92"

In [89]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [90]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [91]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/AdeebInamdar/end-to-end-machine-learning-project-with-mlflow.mlflow",
           
        )

        return model_evaluation_config


In [92]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [95]:
pip install dagshub

  Using cached dagshub-0.6.9-py3-none-any.whl.metadata (12 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.3.1-py3-none-any.whl.metadata (12 kB)
  Using cached boto3-1.42.84-py3-none-any.whl.metadata (6.7 kB)
  Using cached semver-3.0.4-py3-none-any.whl.metadata (6.8 kB)
  Using cached dagshub_annotation_converter-0.1.16-py3-none-any.whl.metadata (3.2 kB)
  Using cached botocore-1.42.84-py3-none-any.whl.metadata (5.9 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata 

In [96]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        import dagshub
        dagshub.init(repo_owner='AdeebInamdar', 
                     repo_name='end-to-end-machine-learning-project-with-mlflow', 
                     mlflow=True)

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [ ]:
# class ModelEvaluation:
#     def __init__(self, config: ModelEvaluationConfig):
#         self.config = config

    
#     def eval_metrics(self,actual, pred):
#         rmse = np.sqrt(mean_squared_error(actual, pred))
#         mae = mean_absolute_error(actual, pred)
#         r2 = r2_score(actual, pred)
#         return rmse, mae, r2
    


#     def log_into_mlflow(self):

#         test_data = pd.read_csv(self.config.test_data_path)
#         model = joblib.load(self.config.model_path)

#         test_x = test_data.drop([self.config.target_column], axis=1)
#         test_y = test_data[[self.config.target_column]]


#         mlflow.set_registry_uri(self.config.mlflow_uri)
#         tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


#         with mlflow.start_run():

#             predicted_qualities = model.predict(test_x)

#             (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
#             # Saving metrics as local
#             scores = {"rmse": rmse, "mae": mae, "r2": r2}
#             save_json(path=Path(self.config.metric_file_name), data=scores)

#             mlflow.log_params(self.config.all_params)

#             mlflow.log_metric("rmse", rmse)
#             mlflow.log_metric("r2", r2)
#             mlflow.log_metric("mae", mae)


#             # Model registry does not work with file store
#             if tracking_url_type_store != "file":

#                 # Register the model
#                 # There are other ways to use the Model Registry, which depends on the use case,
#                 # please refer to the doc for more information:
#                 # https://mlflow.org/docs/latest/model-registry.html#api-workflow
#                 mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
#             else:
#                 mlflow.sklearn.log_model(model, "model")

    


In [97]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-04-07 23:07:17,542: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-07 23:07:17,544: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-07 23:07:17,546: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-07 23:07:17,549: INFO: common: created directory at: artifacts]
[2026-04-07 23:07:17,550: INFO: common: created directory at: artifacts/model_evaluation]


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b30fec03-13a1-47a3-94f6-1db70e9b7f44&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e5af92dbc955851f228d3e00356e8807b566015cece15b25fe46d8bacd469197




[2026-04-07 23:08:03,930: INFO: _client: HTTP Request: POST https://dagshub.com/login/oauth/middleman "HTTP/1.1 200 OK"]


...

[2026-04-07 23:08:09,494: INFO: helpers: Repository AdeebInamdar/end-to-end-machine-learning-project-with-mlflow initialized!]


c:\Users\ADEEB INAMDAR\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ElasticNet from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[2026-04-07 23:08:10,867: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/04/07 23:08:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 23:08:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetModel'.
2026/04/07 23:08:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run welcoming-wolf-229 at: https://dagshub.com/AdeebInamdar/end-to-end-machine-learning-project-with-mlflow.mlflow/#/experiments/0/runs/fdebc21a3b7143a78859c56b23247d40
🧪 View experiment at: https://dagshub.com/AdeebInamdar/end-to-end-machine-learning-project-with-mlflow.mlflow/#/experiments/0
